In [11]:
import nltk
from nltk.corpus import wordnet as wn
import random
import pandas as pd
from collections import defaultdict, Counter
import re
import pandas as pd
import random
from transformers import GPTNeoXForCausalLM, AutoTokenizer
from sae_lens import SAE
import torch
from tqdm.notebook import trange, tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

## Sample From Wordnet

In [3]:
# Download required NLTK data
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /home/adiera/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
# POS tags we care about (exclude adverbs 'r')
pos_map = {'n': 'Noun', 'v': 'Verb', 'a': 'Adjective', 's': 'Adjective (satellite)'}

# Count synsets
counts = {}
for pos, label in pos_map.items():
    if pos != 'r':  # skip adverbs
        counts[label] = len(list(wn.all_synsets(pos)))

# Merge satellite adjectives with adjectives
adj_count = counts.pop('Adjective', 0) + counts.pop('Adjective (satellite)', 0)
counts['Adjective'] = adj_count

# Compute total and percentages
total = sum(counts.values())
distribution = {k: (v, 100 * v / total) for k, v in counts.items()}

# Print results
print("WordNet POS Distribution (no adverbs):")
for k, (count, pct) in distribution.items():
    print(f"{k:10s}: {count:6d} synsets ({pct:.1f}%)")
print(f"Total synsets (no adverbs): {total}")

WordNet POS Distribution (no adverbs):
Noun      :  82115 synsets (65.8%)
Verb      :  13767 synsets (11.0%)
Adjective :  28849 synsets (23.1%)
Total synsets (no adverbs): 124731


In [4]:
def is_valid_word(word):
    """Filter out multi-word terms, proper nouns, and weird edge cases"""
    if not word or len(word) < 3:
        return False
    if '_' in word or '-' in word or '.' in word:
        return False
    if word[0].isupper():  # Skip proper nouns
        return False
    if not word.isalpha():
        return False
    return True

def extract_hypernymy_pairs(max_pairs=300, pos_filter=None):
    """Extract hypernym-hyponym pairs from WordNet (unidirectional: hyponym -> hypernym)"""
    pairs = []
    used_pairs = set()
    
    # Get all synsets
    all_synsets = list(wn.all_synsets())
    random.shuffle(all_synsets)
    
    for synset in all_synsets:
        if len(pairs) >= max_pairs:
            break
            
        # Get hypernyms for this synset
        hypernyms = synset.hypernyms()
        
        for hypernym in hypernyms:
            # Get representative words (lemmas) for both synsets
            hyponym_words = [lemma.name() for lemma in synset.lemmas()]
            hypernym_words = [lemma.name() for lemma in hypernym.lemmas()]
            
            # Filter valid words
            hyponym_words = [w for w in hyponym_words if is_valid_word(w)]
            hypernym_words = [w for w in hypernym_words if is_valid_word(w)]
            
            if not hyponym_words or not hypernym_words:
                continue
                
            # Take the most common word from each synset (first one)
            hyponym = hyponym_words[0]
            hypernym_word = hypernym_words[0]
            
            # Ensure unidirectional - create consistent pair key
            pair_key = (hyponym, hypernym_word)
            if pair_key in used_pairs:
                continue
            used_pairs.add(pair_key)
                
            # Add POS balancing
            pos = synset.pos()
            if pos_filter and pos not in pos_filter:
                continue
                
            pairs.append({
                'hyponym': hyponym,
                'hypernym': hypernym_word,
                'relation': 'hypernymy',
                'pos': pos,
                'hyponym_synset': synset.name(),
                'hypernym_synset': hypernym.name(),
                'direction': 'hyponym_to_hypernym'
            })
            
    
    return pairs[:max_pairs]

def extract_hyponymy_pairs(max_pairs=300, pos_filter=None):
    """Extract hyponym pairs from WordNet (unidirectional: hypernym -> hyponym)"""
    pairs = []
    used_pairs = set()
    
    # Get all synsets
    all_synsets = list(wn.all_synsets())
    random.shuffle(all_synsets)
    
    for synset in all_synsets:
        if len(pairs) >= max_pairs:
            break
            
        # Get hyponyms for this synset
        hyponyms = synset.hyponyms()
        
        for hyponym in hyponyms:
            # Get representative words (lemmas) for both synsets
            hypernym_words = [lemma.name() for lemma in synset.lemmas()]
            hyponym_words = [lemma.name() for lemma in hyponym.lemmas()]
            
            # Filter valid words
            hypernym_words = [w for w in hypernym_words if is_valid_word(w)]
            hyponym_words = [w for w in hyponym_words if is_valid_word(w)]
            
            if not hypernym_words or not hyponym_words:
                continue
                
            # Take the most common word from each synset (first one)
            hypernym_word = hypernym_words[0]
            hyponym_word = hyponym_words[0]
            
            # Ensure unidirectional - create consistent pair key
            pair_key = (hypernym_word, hyponym_word)
            if pair_key in used_pairs:
                continue
            used_pairs.add(pair_key)
                
            # Add POS balancing
            pos = synset.pos()
            if pos_filter and pos not in pos_filter:
                continue
                
            pairs.append({
                'hypernym': hypernym_word,
                'hyponym': hyponym_word,
                'relation': 'hyponymy',
                'pos': pos,
                'hypernym_synset': synset.name(),
                'hyponym_synset': hyponym.name(),
                'direction': 'hypernym_to_hyponym'
            })
    
    return pairs[:max_pairs]

def extract_synonymy_pairs(max_pairs=300, pos_filter=None):
    """Extract synonym pairs from WordNet (unidirectional to avoid duplicates)"""
    pairs = []
    used_pairs = set()
    
    all_synsets = list(wn.all_synsets())
    random.shuffle(all_synsets)
    
    for synset in all_synsets:
        if len(pairs) >= max_pairs:
            break
            
        # Get all lemmas (words) in this synset
        lemmas = [lemma.name() for lemma in synset.lemmas()]
        valid_lemmas = [w for w in lemmas if is_valid_word(w)]
        
        if len(valid_lemmas) < 2:
            continue
            
        pos = synset.pos()
        if pos_filter and pos not in pos_filter:
            continue
            
        # Create pairs from words in the same synset (synonyms)
        # Sort to ensure consistent ordering and avoid duplicates
        valid_lemmas.sort()
        for i in range(len(valid_lemmas)):
            if len(pairs) >= max_pairs:
                break
            for j in range(i+1, len(valid_lemmas)):
                word1, word2 = valid_lemmas[i], valid_lemmas[j]
                
                # Create consistent pair key (alphabetical order)
                pair_key = (word1, word2) if word1 < word2 else (word2, word1)
                if pair_key in used_pairs:
                    continue
                    
                # Skip if words are too similar (likely morphological variants)
                if word1.startswith(word2) or word2.startswith(word1):
                    continue
                    
                used_pairs.add(pair_key)
                    
                pairs.append({
                    'word1': pair_key[0],  # Always alphabetically first
                    'word2': pair_key[1],  # Always alphabetically second
                    'relation': 'synonymy',
                    'pos': pos,
                    'synset': synset.name(),
                    'direction': 'bidirectional'
                })
                
                if len(pairs) >= max_pairs:
                    break
                
    return pairs[:max_pairs]

def extract_antonymy_pairs(max_pairs=300, pos_filter=None):
    """Extract antonym pairs from WordNet (unidirectional to avoid duplicates)"""
    pairs = []
    used_pairs = set()
    processed_lemmas = set()  # Track processed lemmas to avoid duplicates
    
    all_synsets = list(wn.all_synsets())
    random.shuffle(all_synsets)
    
    for synset in all_synsets:
        if len(pairs) >= max_pairs:
            break
            
        pos = synset.pos()
        if pos_filter and pos not in pos_filter:
            continue
            
        # Get all lemmas in this synset
        lemmas = synset.lemmas()
        
        for lemma in lemmas:
            if len(pairs) >= max_pairs:
                break
                
            word1 = lemma.name()
            if not is_valid_word(word1):
                continue
                
            # Skip if we've already processed this lemma
            lemma_key = (word1, lemma.synset().name())
            if lemma_key in processed_lemmas:
                continue
            processed_lemmas.add(lemma_key)
                
            # Get antonyms for this lemma
            antonyms = lemma.antonyms()
            
            for antonym_lemma in antonyms:
                if len(pairs) >= max_pairs:
                    break
                    
                word2 = antonym_lemma.name()
                if not is_valid_word(word2):
                    continue
                    
                # Create consistent pair key (alphabetical order)
                pair_key = (word1, word2) if word1 < word2 else (word2, word1)
                if pair_key in used_pairs:
                    continue
                
                used_pairs.add(pair_key)
                
                pairs.append({
                    'word1': pair_key[0],  # Always alphabetically first
                    'word2': pair_key[1],  # Always alphabetically second
                    'relation': 'antonymy',
                    'pos': pos,
                    'synset1': synset.name(),
                    'synset2': antonym_lemma.synset().name(),
                    'direction': 'bidirectional'
                })
                    
    return pairs[:max_pairs]

def generate_random_pairs(max_pairs=300):
    """Generate random word pairs as control (unidirectional)"""
    # Get vocabulary from WordNet
    vocabulary = []
    for synset in wn.all_synsets():
        for lemma in synset.lemmas():
            word = lemma.name()
            if is_valid_word(word):
                vocabulary.append(word)
    vocabulary = list(set(vocabulary))
    
    pairs = []
    used_pairs = set()
    
    while len(pairs) < max_pairs:
        word1 = random.choice(vocabulary)
        word2 = random.choice(vocabulary)
        
        if word1 == word2:
            continue
            
        # Create consistent pair key (alphabetical order)
        pair_key = (word1, word2) if word1 < word2 else (word2, word1)
        if pair_key in used_pairs:
            continue
        used_pairs.add(pair_key)
            
        # Check that they're not actually related
        if are_words_related(word1, word2):
            continue
            
        pairs.append({
            'word1': pair_key[0],
            'word2': pair_key[1],
            'relation': 'random',
            'pos': 'mixed',
            'direction': 'none'
        })
    
    return pairs

def balance_pairs_by_pos(pairs, target_distribution=None):
    """Balance pairs by part-of-speech"""
    if target_distribution is None:
        target_distribution = {'n': 0.66, 'v': 0.23, 'a': 0.11}
    
    pos_groups = defaultdict(list)
    for pair in pairs:
        pos = pair.get('pos', 'unknown')
        pos_groups[pos].append(pair)
    
    balanced_pairs = []
    total_target = len(pairs)
    
    for pos, proportion in target_distribution.items():
        if pos in pos_groups:
            target_count = int(total_target * proportion)
            available = pos_groups[pos]
            selected = random.sample(available, min(target_count, len(available)))
            balanced_pairs.extend(selected)
    
    return balanced_pairs

def balance_and_fill_pairs(original_data, target_count, relation_type, extract_func):
    """Balance pairs by POS and fill remaining slots"""
    
    # Balance existing data
    balanced_data = balance_pairs_by_pos(original_data)
    remaining = target_count - len(balanced_data)
    
    if remaining > 0:
        print(f"Filling {remaining} more slots for {relation_type}")
        # Extract more data and take what we need
        extra_data = extract_func(remaining * 2)  # Extract extra to account for filtering
        balanced_data.extend(extra_data[:remaining])
    
    return balanced_data[:target_count]

def are_words_related(word1, word2):
    """Simple check if two words might be semantically related"""
    # Get synsets for both words
    synsets1 = wn.synsets(word1)
    synsets2 = wn.synsets(word2)
    
    if not synsets1 or not synsets2:
        return False
    
    # Check for direct relationships
    for s1 in synsets1:
        for s2 in synsets2:
            # Check hypernymy/hyponymy
            if s1 in s2.closure(lambda s: s.hypernyms()):
                return True
            if s2 in s1.closure(lambda s: s.hypernyms()):
                return True
            # Check if they're the same synset (synonyms)
            if s1 == s2:
                return True
            
            # Check antonymy
            for lemma1 in s1.lemmas():
                for lemma2 in s2.lemmas():
                    if lemma2 in lemma1.antonyms():
                        return True
    
    return False
    
def check_duplicates(df):
    normalized_check = set()
    potential_duplicates = 0
    for _, row in df.iterrows():
        pair = tuple(sorted([row['word1'], row['word2']]))
        if pair in normalized_check:
            potential_duplicates += 1
            #print(pair)
            #print(row['relation'])
        else:
            normalized_check.add(pair)

In [5]:
def create_dataset(hypernymy_pairs=300, synonymy_pairs=300, hyponymy_pairs=300, antonymy_pairs=300, random_pairs=300):
    """Create the complete dataset with slot filling"""
    
    print("Extracting hypernymy pairs...")
    hypernymy_data = extract_hypernymy_pairs(hypernymy_pairs, pos_filter=['n', 'v', 'a'])
    
    print("Extracting synonymy pairs...")  
    synonymy_data = extract_synonymy_pairs(synonymy_pairs, pos_filter=['n', 'v', 'a'])
    
    print("Extracting hyponymy pairs...")  
    hyponymy_data = extract_hyponymy_pairs(hyponymy_pairs, pos_filter=['n', 'v', 'a'])
    
    print("Extracting antonym pairs...")  
    antonymy_data = extract_antonymy_pairs(antonymy_pairs, pos_filter=['n', 'v', 'a'])
    
    print("Generating random pairs...")   
    random_data = generate_random_pairs(random_pairs)

    print(len(random_data))
    
    # Balance by POS and fill remaining slots
    print("Balancing by part-of-speech...")
    hypernymy_balanced = balance_and_fill_pairs(hypernymy_data, hypernymy_pairs, 'hypernymy', 
                                               lambda n: extract_hypernymy_pairs(n, pos_filter=['n', 'v', 'a']))
    
    synonymy_balanced = balance_and_fill_pairs(synonymy_data, synonymy_pairs, 'synonymy',
                                              lambda n: extract_synonymy_pairs(n, pos_filter=['n', 'v', 'a']))
    
    hyponymy_balanced = balance_and_fill_pairs(hyponymy_data, hyponymy_pairs, 'hyponymy',
                                              lambda n: extract_hyponymy_pairs(n, pos_filter=['n', 'v', 'a']))
    
    antonymy_balanced = balance_and_fill_pairs(antonymy_data, antonymy_pairs, 'antonymy',
                                              lambda n: extract_antonymy_pairs(n, pos_filter=['n', 'v', 'a']))
    
    print(f"Final counts: hypernymy={len(hypernymy_balanced)}, synonymy={len(synonymy_balanced)}, "
          f"hyponymy={len(hyponymy_balanced)}, antonymy={len(antonymy_balanced)}, random={len(random_data)}")
    
    # Create standardized format
    dataset = []
    
    # Add hypernymy pairs
    for pair in hypernymy_balanced:
        dataset.append({
            'word1': pair['hyponym'],
            'word2': pair['hypernym'], 
            'relation': 'hypernymy',
            'direction': 'hyponym_to_hypernym',
            'pos': pair['pos']
        })
    
    # Add hyponymy pairs
    for pair in hyponymy_balanced: 
        dataset.append({
            'word1': pair['hypernym'],
            'word2': pair['hyponym'],
            'relation': 'hyponymy', 
            'direction': 'hypernym_to_hyponym',
            'pos': pair['pos']
        })
    
    # Add synonymy pairs
    for pair in synonymy_balanced:
        dataset.append({
            'word1': pair['word1'],
            'word2': pair['word2'],
            'relation': 'synonymy',
            'direction': 'bidirectional', 
            'pos': pair['pos']
        })
    
    # Add antonymy pairs
    for pair in antonymy_balanced:
        dataset.append({
            'word1': pair['word1'],
            'word2': pair['word2'],
            'relation': 'antonymy',
            'direction': 'bidirectional', 
            'pos': pair['pos']
        })
    
    # Add random pairs
    for pair in random_data:
        dataset.append({
            'word1': pair['word1'],
            'word2': pair['word2'],
            'relation': 'random',
            'direction': 'none',
            'pos': pair['pos']  
        })
    
    return dataset

def remove_duplicates_and_refill(df_dataset):
    """Remove duplicate word pairs and refill with new samples"""
    
    # Track seen pairs and find duplicates
    seen_pairs = set()
    duplicate_indices = []
    
    for idx, row in df_dataset.iterrows():
        pair = tuple(sorted([row['word1'], row['word2']]))
        if pair in seen_pairs:
            duplicate_indices.append(idx)
            print(f"Duplicate found: {pair} in {row['relation']}")
        else:
            seen_pairs.add(pair)
    
    if not duplicate_indices:
        print("No duplicates found!")
        return remove_duplicates_and_refill(dataset)
    
    print(f"Found {len(duplicate_indices)} duplicates, removing and refilling...")
    
    # Group duplicates by relation type
    duplicates_by_relation = {}
    for idx in duplicate_indices:
        relation = df_dataset.loc[idx, 'relation']
        if relation not in duplicates_by_relation:
            duplicates_by_relation[relation] = []
        duplicates_by_relation[relation].append(idx)
    
    # Remove duplicates
    df_clean = df_dataset.drop(duplicate_indices).reset_index(drop=True)
    
    # Refill each relation type
    for relation, indices in duplicates_by_relation.items():
        needed_count = len(indices)
        pos_needed = [df.loc[idx, 'pos'] for idx in indices]  # Keep same POS distribution
        
        print(f"Refilling {needed_count} {relation} pairs...")
        
        # Generate new samples
        new_samples = refill_new_samples_for_relation(relation, needed_count, pos_needed, seen_pairs)
        
        # Add new samples to dataset
        for sample in new_samples:
            new_row = pd.DataFrame([sample])
            df_clean = pd.concat([df_clean, new_row], ignore_index=True)
            
            # Update seen pairs
            pair = tuple(sorted([sample['word1'], sample['word2']]))
            seen_pairs.add(pair)
    
    return df_clean

def refill_new_samples_for_relation(relation, count, pos_list, existing_pairs):
    """Generate new samples for a specific relation, avoiding duplicates"""
    
    new_samples = []
    attempts = 0
    max_attempts = count * 10  # Prevent infinite loops
    
    while len(new_samples) < count and attempts < max_attempts:
        attempts += 1
        
        # Generate based on relation type
        if relation == 'hypernymy':
            candidates = extract_hypernymy_pairs(count * 2, pos_filter=['n', 'v', 'a'])
            for candidate in candidates:
                if len(new_samples) >= count:
                    break
                pair = tuple(sorted([candidate['hyponym'], candidate['hypernym']]))
                if pair not in existing_pairs:
                    new_samples.append({
                        'word1': candidate['hyponym'],
                        'word2': candidate['hypernym'],
                        'relation': 'hypernymy',
                        'direction': 'hyponym_to_hypernym',
                        'pos': candidate['pos']
                    })
                    existing_pairs.add(pair)
        
        elif relation == 'hyponymy':
            candidates = extract_hyponymy_pairs(count * 2, pos_filter=['n', 'v', 'a'])
            for candidate in candidates:
                if len(new_samples) >= count:
                    break
                pair = tuple(sorted([candidate['hypernym'], candidate['hyponym']]))
                if pair not in existing_pairs:
                    new_samples.append({
                        'word1': candidate['hypernym'],
                        'word2': candidate['hyponym'],
                        'relation': 'hyponymy',
                        'direction': 'hypernym_to_hyponym',
                        'pos': candidate['pos']
                    })
                    existing_pairs.add(pair)
        
        elif relation == 'synonymy':
            candidates = extract_synonymy_pairs(count * 2, pos_filter=['n', 'v', 'a'])
            for candidate in candidates:
                if len(new_samples) >= count:
                    break
                pair = tuple(sorted([candidate['word1'], candidate['word2']]))
                if pair not in existing_pairs:
                    new_samples.append({
                        'word1': candidate['word1'],
                        'word2': candidate['word2'],
                        'relation': 'synonymy',
                        'direction': 'bidirectional',
                        'pos': candidate['pos']
                    })
                    existing_pairs.add(pair)
        
        elif relation == 'antonymy':
            candidates = extract_antonymy_pairs(count * 2, pos_filter=['n', 'v', 'a'])
            for candidate in candidates:
                if len(new_samples) >= count:
                    break
                pair = tuple(sorted([candidate['word1'], candidate['word2']]))
                if pair not in existing_pairs:
                    new_samples.append({
                        'word1': candidate['word1'],
                        'word2': candidate['word2'],
                        'relation': 'antonymy',
                        'direction': 'bidirectional',
                        'pos': candidate['pos']
                    })
                    existing_pairs.add(pair)
        
        elif relation == 'random':
            candidates = generate_random_pairs(count * 2, pos_filter=['n', 'v', 'a'])
            for candidate in candidates:
                if len(new_samples) >= count:
                    break
                pair = tuple(sorted([candidate['word1'], candidate['word2']]))
                if pair not in existing_pairs:
                    new_samples.append({
                        'word1': candidate['word1'],
                        'word2': candidate['word2'],
                        'relation': 'random',
                        'direction': 'none',
                        'pos': candidate['pos']
                    })
                    existing_pairs.add(pair)
    
    if len(new_samples) < count:
        print(f"Warning: Only generated {len(new_samples)} out of {count} requested {relation} samples")
    
    return new_samples

In [13]:
print("Creating enhanced semantic relation dataset...")
auto_dataset = create_dataset(1000, 1000, 1000, 1000, 1000)  # hypernymy, hyponymy, synonymy, antonymy, random
df = pd.DataFrame(auto_dataset)

# Create final DataFrame
df = pd.DataFrame(auto_dataset)
# Remove duplicates and refill
df = remove_duplicates_and_refill(df)

# Check for any potential remaining duplicates
check_duplicates(df)

# Print statistics
print(f"\nFinal Dataset Statistics:")
print(f"Total pairs: {len(df)}")
print(f"\nRelation distribution:")
print(df['relation'].value_counts())
print(f"\nDirection distribution:")
print(df['direction'].value_counts())

# Save datasets
df.to_csv('./dataset/semantic_relations.csv', index=False)

print(f"\nDatasets saved:")
print(f"  - semantic_relations.csv (automated)")

# Show sample pairs
print(f"\nSample pairs from pilot dataset:")
for relation in ['hypernymy', 'hyponymy', 'synonymy', 'antonymy', 'random']:
    if relation in df['relation'].values:
        print(f"\n{relation.upper()} examples:")
        samples = df[df['relation'] == relation].sample(min(5, len(df[df['relation'] == relation])))
        for _, row in samples.iterrows():
            print(f"  {row['word1']} → {row['word2']} ({row['direction']})")

# Export specific formats for analysis
print(f"\nExporting analysis-ready formats...")

# Hypernymy pairs only
hyp_pairs = df[df['relation'] == 'hypernymy'][['word1', 'word2', 'direction']].copy()
hyp_pairs.to_csv('./dataset/hypernymy_pairs".csv', index=False)

# Hyponymy pairs only
hypo_pairs = df[df['relation'] == 'hyponymy'][['word1', 'word2', 'direction']].copy()
hypo_pairs.to_csv('./dataset/hyponymy_pairs.csv', index=False)

# Synonymy pairs only  
syn_pairs = df[df['relation'] == 'synonymy'][['word1', 'word2']].copy()
syn_pairs.to_csv('./dataset/synonymy_pairs.csv', index=False)

# Antonymy pairs only
ant_pairs = df[df['relation'] == 'antonymy'][['word1', 'word2']].copy()
ant_pairs.to_csv('./dataset/antonymy_pairs.csv', index=False)

# Random pairs only
rand_pairs = df[df['relation'] == 'random'][['word1', 'word2']].copy() 
rand_pairs.to_csv('./dataset/random_pairs.csv', index=False)

print("Individual relation files created:")
print("  - hypernymy_pairs.csv")
print("  - hyponymy_pairs.csv") 
print("  - synonymy_pairs.csv")
print("  - antonymy_pairs.csv")
print("  - random_pairs.csv")

# Additional analysis
print(f"\nAdditional Analysis:")
print(f"Unique words in dataset: {len(set(df['word1'].tolist() + df['word2'].tolist()))}")

Creating enhanced semantic relation dataset...
Generating automated dataset...
Extracting hypernymy pairs...
Extracting synonymy pairs...
Extracting hyponymy pairs...
Extracting antonym pairs...
Generating random pairs...
1000
Balancing by part-of-speech...
Filling 110 more slots for hypernymy
Filling 84 more slots for synonymy
Filling 170 more slots for hyponymy
Filling 479 more slots for antonymy
Final counts: hypernymy=1000, synonymy=1000, hyponymy=1000, antonymy=1000, random=1000
Duplicate found: ('leper', 'outcast') in hypernymy
Duplicate found: ('letterpress', 'printing') in hypernymy
Duplicate found: ('photograph', 'telephotograph') in hypernymy
Duplicate found: ('contact', 'engagement') in hypernymy
Duplicate found: ('corrosion', 'pitting') in hypernymy
Duplicate found: ('scenery', 'vicinity') in hyponymy
Duplicate found: ('murmur', 'symptom') in hyponymy
Duplicate found: ('garment', 'sweater') in hyponymy
Duplicate found: ('garment', 'veil') in hyponymy
Duplicate found: ('entr

## Base Train Test Sets

In [5]:
def create_train_test_split(df, test_size=0.2, random_state=42):
    
    # Check current class distribution
    print("Original class distribution:")
    print(df['relation'].value_counts())
    print(f"Total samples: {len(df)}")
    
    # Stratified split by relation type
    train_df, test_df = train_test_split(
        df, 
        test_size=test_size, 
        stratify=df['relation'],
        random_state=random_state
    )
    
    print(f"\nTrain set: {len(train_df)} samples")
    print(train_df['relation'].value_counts())
    
    print(f"\nTest set: {len(test_df)} samples") 
    print(test_df['relation'].value_counts())
    
    return train_df, test_df

In [16]:
train_df, test_df = create_train_test_split(df)

Original class distribution:
hypernymy    1000
hyponymy     1000
synonymy     1000
antonymy     1000
random       1000
Name: relation, dtype: int64
Total samples: 5000

Train set: 4000 samples
hypernymy    800
synonymy     800
random       800
antonymy     800
hyponymy     800
Name: relation, dtype: int64

Test set: 1000 samples
hypernymy    200
random       200
antonymy     200
hyponymy     200
synonymy     200
Name: relation, dtype: int64


In [19]:
# Define prompt templates
context_templates = [
    "The word {A} relates to {B}",
    "{A} and {B} are connected",
    "Consider {A} and {B} together",
    #"{A} {B}"  # Just the two words side by side
]

# Create prompts
prompts = []
for _, row in train_df.iterrows():
    word1, word2 = row['word1'], row['word2']
    for template in context_templates:
        prompt = template.format(A=word1, B=word2)
        prompts.append({
            "prompt": prompt,
            "relation": row['relation']
        })

# Convert to DataFrame and save
prompt_df = pd.DataFrame(prompts)
prompt_df.to_csv('./dataset/semantic_relation_neutral_prompts_context_train.csv', index=False)

print(prompt_df.head())

prompts = []
for _, row in test_df.iterrows():
    word1, word2 = row['word1'], row['word2']
    for template in context_templates:
        prompt = template.format(A=word1, B=word2)
        prompts.append({
            "prompt": prompt,
            "relation": row['relation']
        })

# Convert to DataFrame and save
prompt_df = pd.DataFrame(prompts)
prompt_df.to_csv('./dataset/semantic_relation_neutral_prompts_context_test.csv', index=False)

print(prompt_df.head())

                                        prompt   relation
0    The word heterocyclic relates to compound  hypernymy
1      heterocyclic and compound are connected  hypernymy
2  Consider heterocyclic and compound together  hypernymy
3        The word vocalise relates to vocalize   synonymy
4          vocalise and vocalize are connected   synonymy
                                            prompt   relation
0                 The word slip relates to mistake  hypernymy
1                   slip and mistake are connected  hypernymy
2               Consider slip and mistake together  hypernymy
3  The word geometrically relates to hematopoietic     random
4    geometrically and hematopoietic are connected     random


## Hypernym & Hyponym Flipping Sets

In [28]:
def create_train_test_split_hyper_hypo(df, test_size=0.2, random_state=42):
    #  drop random
    df = (df[df['relation'].isin(['hypernymy','hyponymy','random'])].reset_index(drop=True).copy())
    
    # Check current class distribution
    print("Filtered class distribution:")
    print(df['relation'].value_counts())
    print(f"Total samples: {len(df)}")
    
    # Stratified split by relation type
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df['relation'],
        random_state=random_state
    )
    
    print(f"\nTrain set: {len(train_df)} samples")
    print(train_df['relation'].value_counts())
    
    print(f"\nTest set: {len(test_df)} samples")
    print(test_df['relation'].value_counts())
    
    return train_df, test_df

In [30]:
df = pd.read_csv('./dataset/semantic_relations.csv')
train_hh_df, test_hh_df = create_train_test_split_hyper_hypo(df)

Filtered class distribution:
hypernymy    1000
hyponymy     1000
synonymy     1000
antonymy     1000
Name: relation, dtype: int64
Total samples: 4000

Train set: 3200 samples
synonymy     800
hyponymy     800
antonymy     800
hypernymy    800
Name: relation, dtype: int64

Test set: 800 samples
antonymy     200
hypernymy    200
synonymy     200
hyponymy     200
Name: relation, dtype: int64


In [31]:
# Define prompt templates
context_templates = [
    "The word {A} relates to {B}",
    "{A} and {B} are connected",
    "Consider {A} and {B} together"
]

# -----------------------------
# Train set (original only)
# -----------------------------
train_prompts = []
for _, row in train_hh_df.iterrows():
    w1, w2, rel = row['word1'], row['word2'], row['relation']
    for template in context_templates:
        train_prompts.append({
            "prompt": template.format(A=w1, B=w2),
            "relation": rel
        })

train_prompt_df = pd.DataFrame(train_prompts)
train_prompt_df.to_csv('./dataset/hyp_relation_neutral_prompts_context_train.csv', index=False)
print("Train set:\n", train_prompt_df['relation'].value_counts())

# -----------------------------
# Test set (original order)
# -----------------------------
test_prompts_original = []
for _, row in test_hh_df.iterrows():
    w1, w2, rel = row['word1'], row['word2'], row['relation']
    for template in context_templates:
        test_prompts_original.append({
            "prompt": template.format(A=w1, B=w2),
            "relation": rel
        })

test_prompt_df_orig = pd.DataFrame(test_prompts_original)
test_prompt_df_orig.to_csv('./dataset/hyp_relation_neutral_prompts_context_test_original.csv', index=False)
print("Original test set:\n", test_prompt_df_orig['relation'].value_counts())

# -----------------------------
# Test set (flipped order + flipped labels)
# -----------------------------
test_prompts_flipped = []

# Only flip these two; everything else stays the same
flip_map = {
    "hypernymy": "hyponymy",
    "hyponymy": "hypernymy",
}

for _, row in test_hh_df.iterrows():
    w1, w2, rel = row['word1'], row['word2'], row['relation']

    # Flip label only if it's hypernymy/hyponymy; otherwise keep as-is
    flipped_rel = flip_map.get(rel, rel)

    for template in context_templates:
        test_prompts_flipped.append({
            "prompt": template.format(A=w2, B=w1),  # always swap word order
            "relation": flipped_rel                 # antonymy/synonymy unchanged
        })

test_prompt_df_flip = pd.DataFrame(test_prompts_flipped)
test_prompt_df_flip.to_csv('./dataset/hyp_relation_neutral_prompts_context_test_flipped.csv', index=False)
print("Flipped test set:\n", test_prompt_df_flip['relation'].value_counts())

Train set:
 synonymy     2400
hyponymy     2400
antonymy     2400
hypernymy    2400
Name: relation, dtype: int64
Original test set:
 antonymy     600
hypernymy    600
synonymy     600
hyponymy     600
Name: relation, dtype: int64
Flipped test set:
 antonymy     600
hyponymy     600
synonymy     600
hypernymy    600
Name: relation, dtype: int64


In [32]:
# Load both test CSVs you created earlier
orig_df = pd.read_csv('./dataset/hyp_relation_neutral_prompts_context_test_original.csv')
flip_df = pd.read_csv('./dataset/hyp_relation_neutral_prompts_context_test_flipped.csv')

# Sanity check: sample a few rows side-by-side
print("\nSanity check (first 10 examples):\n")
for i in range(10):
    orig = orig_df.iloc[i]
    flip = flip_df.iloc[i]
    print(f"Original: {orig['prompt']} --> {orig['relation']}")
    print(f"Flipped:  {flip['prompt']} --> {flip['relation']}\n")


Sanity check (first 10 examples):

Original: The word agreeableness relates to disagreeableness --> antonymy
Flipped:  The word disagreeableness relates to agreeableness --> antonymy

Original: agreeableness and disagreeableness are connected --> antonymy
Flipped:  disagreeableness and agreeableness are connected --> antonymy

Original: Consider agreeableness and disagreeableness together --> antonymy
Flipped:  Consider disagreeableness and agreeableness together --> antonymy

Original: The word text relates to matter --> hypernymy
Flipped:  The word matter relates to text --> hyponymy

Original: text and matter are connected --> hypernymy
Flipped:  matter and text are connected --> hyponymy

Original: Consider text and matter together --> hypernymy
Flipped:  Consider matter and text together --> hyponymy

Original: The word bind relates to unbind --> antonymy
Flipped:  The word unbind relates to bind --> antonymy

Original: bind and unbind are connected --> antonymy
Flipped:  unbind 

## Context Robustness

In [8]:
df = pd.read_csv('./dataset/semantic_relations.csv')
train_r_df, test_r_df = create_train_test_split(df)

Original class distribution:
hypernymy    1000
hyponymy     1000
synonymy     1000
antonymy     1000
random       1000
Name: relation, dtype: int64
Total samples: 5000

Train set: 4000 samples
hypernymy    800
synonymy     800
random       800
antonymy     800
hyponymy     800
Name: relation, dtype: int64

Test set: 1000 samples
hypernymy    200
random       200
antonymy     200
hyponymy     200
synonymy     200
Name: relation, dtype: int64


In [10]:
# Define prompt templates
context_templates_orig = [
    "The word {A} relates to {B}",
    "{A} and {B} are connected",
    "Consider {A} and {B} together",
    #"{A} {B}"  # Just the two words side by side
]

context_templates_novel = [
    "{A} occurs with {B}"
]

context_templates_nothing = [
    "{A} {B}"  # Just the two words side by side
]

# Create prompts
prompts = []
for _, row in train_r_df.iterrows():
    word1, word2 = row['word1'], row['word2']
    for template in context_templates_orig:
        prompt = template.format(A=word1, B=word2)
        prompts.append({
            "prompt": prompt,
            "relation": row['relation']
        })

# Convert to DataFrame and save
prompt_df = pd.DataFrame(prompts)
prompt_df.to_csv('./dataset/semantic_relation_neutral_prompts_context_train.csv', index=False)

print(prompt_df.head())

prompts = []
for _, row in test_r_df.iterrows():
    word1, word2 = row['word1'], row['word2']
    for template in context_templates_novel:
        prompt = template.format(A=word1, B=word2)
        prompts.append({
            "prompt": prompt,
            "relation": row['relation']
        })

# Convert to DataFrame and save
prompt_df = pd.DataFrame(prompts)
prompt_df.to_csv('./dataset/semantic_relation_novel_prompts_context_test.csv', index=False)

print(prompt_df.head())

prompts = []
for _, row in test_r_df.iterrows():
    word1, word2 = row['word1'], row['word2']
    for template in context_templates_nothing:
        prompt = template.format(A=word1, B=word2)
        prompts.append({
            "prompt": prompt,
            "relation": row['relation']
        })

# Convert to DataFrame and save
prompt_df = pd.DataFrame(prompts)
prompt_df.to_csv('./dataset/semantic_relation_no_prompts_context_test.csv', index=False)

print(prompt_df.head())

                                        prompt   relation
0    The word heterocyclic relates to compound  hypernymy
1      heterocyclic and compound are connected  hypernymy
2  Consider heterocyclic and compound together  hypernymy
3        The word vocalise relates to vocalize   synonymy
4          vocalise and vocalize are connected   synonymy
                                    prompt   relation
0                 slip occurs with mistake  hypernymy
1  geometrically occurs with hematopoietic     random
2              ecdemic occurs with endemic   antonymy
3                worker occurs with tacker   hyponymy
4                   stew occurs with bigos   hyponymy
                        prompt   relation
0                 slip mistake  hypernymy
1  geometrically hematopoietic     random
2              ecdemic endemic   antonymy
3                worker tacker   hyponymy
4                   stew bigos   hyponymy
